# Uso de GPU

In [22]:
import tensorflow as tf

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU {i} details:", details)
else:
    print("Nenhuma GPU visível para o TensorFlow.")

I0000 00:00:1780078757.354344 1242472 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
Built with CUDA: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU 0 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}
GPU 1 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}


# Análise demográfica

In [23]:
import pandas as pd

df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

Conjuntos (pacientes com 3 linhas): 525
Linhas totais: 1575
Checagem linhas/3: 525.0

Por GROUP:
GROUP
pMCI    128
sMCI    397
Name: count, dtype: int64

Por SEX:
SEX
F    222
M    303
Name: count, dtype: int64

GROUP × SEX:
SEX      F    M  All
GROUP               
pMCI    54   74  128
sMCI   168  229  397
All    222  303  525


In [24]:
df_ext.groupby("GROUP")["ID_PT"].nunique()

GROUP
pMCI    128
sMCI    397
Name: ID_PT, dtype: int64

In [25]:
dist_pacientes = (
    df_ext.drop_duplicates(subset=["ID_PT", "GROUP"])
    .groupby(["GROUP", "SEX"])["ID_PT"]
    .nunique()
    .unstack(fill_value=0)
)

dist_pacientes

SEX,F,M
GROUP,,
pMCI,54,74
sMCI,168,229


In [26]:
pts = df_ext.groupby(["GROUP", "ID_PT"], as_index=False)["AGE"].min()

for g in ["sMCI", "pMCI"]:
    a = pts.loc[pts["GROUP"] == g, "AGE"]
    print(
        f"{g}: n={a.count()}, "
        f"min={a.min():.1f}, max={a.max():.1f}, "
        f"media={a.mean():.2f}, desvio={a.std():.2f}"
    )

sMCI: n=397, min=55.0, max=91.0, media=73.52, desvio=7.48
pMCI: n=128, min=57.0, max=89.0, media=75.04, desvio=7.01


In [27]:
adnimerge = pd.read_csv("csvs/adnimerged.csv")
df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

cols = ["ID_IMG", "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE"]

adni_sub = adnimerge.loc[:, cols].copy()
adni_sub["ID_IMG"] = adni_sub["ID_IMG"].astype(str).str.strip()
adni_sub = adni_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

df_ext["ID_IMG"] = df_ext["ID_IMG"].astype(str).str.strip()
df_ext = df_ext.merge(adni_sub, on="ID_IMG", how="left", validate="many_to_one")

In [28]:
pts = (
    df_ext.sort_values(["GROUP", "ID_PT", "MRI_DATE"])
    .drop_duplicates(subset=["GROUP", "ID_PT"], keep="first")
)

pts.groupby("GROUP")[["MMSE_SCORE", "FAQ_SCORE", "CDR_GLOBAL", "ADAS_SCORE"]].agg(["mean", "std", "count"])

MMSE_SCORE                 FAQ_SCORE                 CDR_GLOBAL  \
            mean       std count      mean       std count       mean   
GROUP                                                                   
pMCI   26.039062  2.249768   128  6.453125  4.619131   128   0.507812   
sMCI   27.758186  1.822132   397  2.503778  3.573628   397   0.497481   

                      ADAS_SCORE                  
            std count       mean       std count  
GROUP                                             
pMCI   0.062253   128  14.382891  5.055354   128  
sMCI   0.035444   397   9.491788  3.917522   397

# Testes TODO

- estudar o treinamento dos modelos, se é por épocas ou tudo junto
- verificar como detectar overfitting para o modelo utilizado
- documentar o pipeline
- testar outros modelos, xgboost, random forest
- teste com todos os atributos
- filtro variancia + correlação
- filtro atributos monotonicos
- atributos radiomicos + deslocamento
- estudar wandb


In [23]:
# Nested CV 5×5 — comparação de modelos (pipeline unificado)
# Ordem fiel ao notebook: corr → var → kbest → scaler → classificador

import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message=".*greater than n_features.*",
    category=UserWarning,
    module="sklearn.feature_selection._univariate_selection",
)
# ==========================================
# CONFIGURAÇÕES
# ==========================================
SEED = 42
K_OUT, K_IN = 5, 5
PATH = "csvs/abordagem_4_sMCI_pMCI_extremos/feat_temporal_hipp.csv"
TEST_NAME = "nested_cv_multimodel"

METRIC_COLS = [
    "accuracy","auc", "auc_pr", "bal_acc", "mcc",
    "sens_pMCI", "spec_sMCI", "prec_pMCI", "f1_pMCI",
    "recall_pMCI", "precision_pMCI",
]

# ==========================================
# FILTRO DE CORRELAÇÃO (igual ao notebook)
# ==========================================
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.to_keep_ = None

    def fit(self, X, y=None):
        xf = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        if xf.shape[0] < 2:
            self.to_keep_ = np.arange(X.shape[1], dtype=int)
            return self

        c = np.corrcoef(xf.T)
        keep = []
        for j in range(c.shape[0]):
            if all(
                not (np.isfinite(c[j, k]) and abs(c[j, k]) > self.threshold)
                for k in keep
            ):
                keep.append(j)
        self.to_keep_ = np.asarray(keep, dtype=int)
        return self

    def transform(self, X):
        return X[:, self.to_keep_]


# ==========================================
# MÉTRICAS
# ==========================================
def fold_metrics(y_true, scores, pred) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "auc": float(roc_auc_score(y_true, scores)),
        "auc_pr": float(average_precision_score(y_true, scores)),
        "bal_acc": float(balanced_accuracy_score(y_true, pred)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "sens_pMCI": float(tp / (tp + fn)) if (tp + fn) else float("nan"),
        "spec_sMCI": float(tn / (tn + fp)) if (tn + fp) else float("nan"),
        "prec_pMCI": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
        "f1_pMCI": float(f1_score(y_true, pred, pos_label=1, zero_division=0)),
        "recall_pMCI": float(recall_score(y_true, pred, pos_label=1, zero_division=0)),
        "precision_pMCI": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
    }


def tune_youden_threshold(y_true, scores) -> float:
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(np.unique(y_true)) < 2:
        return 0.0
    fpr, tpr, thr = roc_curve(y_true, scores)
    j = tpr - fpr
    return float(thr[int(np.argmax(j))])


def get_scores(model, X):
    clf = model.named_steps["clf"]
    if hasattr(clf, "decision_function"):
        return model.decision_function(X)
    return model.predict_proba(X)[:, 1]


# def count_features_after_preprocess(model, X):
#     preprocess = Pipeline(model.steps[:-1])
#     return int(preprocess.transform(X).shape[1])
def count_features_after_preprocess(model, X):
    Xt = X
    for name, step in model.steps[:-1]:  # todos exceto o classificador
        if hasattr(step, "transform"):
            Xt = step.transform(Xt)
        # RandomUnderSampler e similares: ignorar (não mudam n_features)
    return int(Xt.shape[1])

def simplify_best_params(best_params: dict) -> dict:
    out = {}
    for k, v in best_params.items():
        if k == "clf":
            out["model"] = v.__class__.__name__
        elif k == "kbest" and v == "passthrough":
            out["kbest"] = "off"
        elif k == "kbest":
            out["kbest"] = "on"
        else:
            out[k.replace("clf__", "").replace("kbest__", "k=")] = v
    return out


# ==========================================
# NESTED CV — COMPARAÇÃO DE MODELOS
# ==========================================
def nested_cv_multimodel(X, y, test_name: str = TEST_NAME):
    results = []

    # base_pipeline = Pipeline([
    #     ("corr_filter", CorrelationFilter(threshold=0.95)),
    #     ("var_filter", VarianceThreshold(threshold=0.01)),
    #     ("kbest", "passthrough"),
    #     ("scaler", StandardScaler()),
    #     ("clf", LinearSVC(random_state=SEED)),  # placeholder
    # ])
    base_pipeline = ImbPipeline([
        ("corr_filter", CorrelationFilter(threshold=0.95)),
        ("var_filter", VarianceThreshold(threshold=0.01)),
        # --- DOWNSAMPLING ENTRA AQUI ---
        ("sampler", RandomUnderSampler(random_state=SEED)), 
        ("kbest", "passthrough"),
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(random_state=SEED)),  # placeholder
    ])
    outer_cv = StratifiedKFold(K_OUT, shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        pos_weight = float(np.sum(y_train == 0) / np.sum(y_train == 1))
        svm = LinearSVC(
            penalty="l1",
            dual=False,
            class_weight="balanced",
            max_iter=200_000,
            random_state=SEED,
        )

        param_grid = [
            # 1) LinearSVC — réplica do Teste 1 (sem SelectKBest)
            # {
            #     "kbest": ["passthrough"],
            #     "clf": [svm],
            #     "clf__C": [1e-3, 0.01, 0.1, 1.0, 10.0],
            # },
            # # 2) LinearSVC + SelectKBest
            # {
            #     "kbest": [SelectKBest(score_func=f_classif)],
            #     "kbest__k": [30, 50, 100, 200],
            #     "clf": [svm],
            #     "clf__C": [1e-3, 0.01, 0.1, 1.0, 10.0],
            # },
            # 3) Random Forest sem SelectKBest
            # {
            #     "kbest": ["passthrough"],
            #     "clf": [RandomForestClassifier(class_weight="balanced", random_state=SEED)],
            #     "clf__n_estimators": [100, 200],
            #     "clf__max_depth": [None, 5, 10],
            # },
            # 4) Random Forest com SelectKBest
            {
                "kbest": [SelectKBest(score_func=f_classif)],
                "kbest__k": [50, 100, 200],
                "clf": [RandomForestClassifier(class_weight="balanced", random_state=SEED)],
                "clf__n_estimators": [100, 200],
                "clf__max_depth": [None, 5, 10],
            },
            # 5) XGBoost sem SelectKBest
            # {
            #     "kbest": ["passthrough"],
            #     "clf": [XGBClassifier(
            #         # scale_pos_weight=pos_weight,
            #         eval_metric="logloss",
            #         random_state=SEED,
            #     )],
            #     "clf__n_estimators": [100, 200],
            #     "clf__learning_rate": [0.01, 0.1],
            #     "clf__max_depth": [3, 5],
            # },
            # # 6) XGBoost com SelectKBest
            # {
            #     "kbest": [SelectKBest(score_func=f_classif)],
            #     "kbest__k": [50, 100, 200],
            #     "clf": [XGBClassifier(
            #         scale_pos_weight=pos_weight,
            #         eval_metric="logloss",
            #         random_state=SEED,
            #     )],
            #     "clf__n_estimators": [100, 200],
            #     "clf__learning_rate": [0.01, 0.1],
            #     "clf__max_depth": [3, 5],
            # },
        ]

        inner_cv = StratifiedKFold(K_IN, shuffle=True, random_state=SEED + fold)

        grid_search = GridSearchCV(
            estimator=base_pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            refit=True,
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        model_name = best_model.named_steps["clf"].__class__.__name__

        # Limiar Youden com scores out-of-fold (sem vazar teste)
        if hasattr(best_model.named_steps["clf"], "decision_function"):
            train_scores = cross_val_predict(
                best_model, X_train, y_train, cv=inner_cv, method="decision_function"
            )
        else:
            train_scores = cross_val_predict(
                best_model, X_train, y_train, cv=inner_cv, method="predict_proba"
            )[:, 1]

        threshold = tune_youden_threshold(y_train, train_scores)

        test_scores = get_scores(best_model, X_test)
        test_preds = (test_scores >= threshold).astype(int)

        row = {
            "test": test_name,
            "fold": fold,
            "best_model": model_name,
            "best_inner_auc": float(grid_search.best_score_),
            "best_params": simplify_best_params(grid_search.best_params_),
            "threshold": threshold,
            "n_features": count_features_after_preprocess(best_model, X_train),
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_test_pMCI": int(y_test.sum()),
            **fold_metrics(y_test, test_scores, test_preds),
        }
        results.append(row)

        print(
            f"Fold {fold} | vencedor: {model_name} | "
            f"AUC interno: {row['best_inner_auc']:.3f} | "
            f"AUC teste: {row['auc']:.3f}"
        )

    return pd.DataFrame(results)


def print_summary(df: pd.DataFrame) -> None:
    print(f"\n=== {df['test'].iloc[0]} ===")
    display(df)

    print("\nResumo geral (média ± desvio entre folds):")
    for col in METRIC_COLS:
        print(f"  {col:14s}: {df[col].mean():.3f} ± {df[col].std():.3f}")

    print("\nModelo vencedor por fold:")
    display(df.groupby("best_model").size().rename("n_folds").to_frame())

    print("\nMédia de AUC no teste por modelo vencedor:")
    display(
        df.groupby("best_model")["auc"]
        .agg(["count", "mean", "std"])
        .rename(columns={"count": "n_folds", "mean": "auc_mean", "std": "auc_std"})
    )

In [24]:
# Carregar dados e rodar
meta = ["ID_PT", "GROUP", "SEX"]
df = pd.read_csv(PATH)

X = df.drop(columns=meta).to_numpy(float)
y = df["GROUP"].to_numpy(int)

print(f"{TEST_NAME} | {len(y)} pacientes | {X.shape[1]} features originais")
print(f"Classes: sMCI={(y == 0).sum()} | pMCI={(y == 1).sum()}\n")

df_results = nested_cv_multimodel(X, y, test_name=TEST_NAME)
print_summary(df_results)

nested_cv_multimodel | 525 pacientes | 1368 features originais
Classes: sMCI=397 | pMCI=128

Fold 1 | vencedor: RandomForestClassifier | AUC interno: 0.755 | AUC teste: 0.761
Fold 2 | vencedor: RandomForestClassifier | AUC interno: 0.773 | AUC teste: 0.727
Fold 3 | vencedor: RandomForestClassifier | AUC interno: 0.759 | AUC teste: 0.858
Fold 4 | vencedor: RandomForestClassifier | AUC interno: 0.725 | AUC teste: 0.770
Fold 5 | vencedor: RandomForestClassifier | AUC interno: 0.772 | AUC teste: 0.732

=== nested_cv_multimodel ===


,test,fold,best_model,best_inner_auc,best_params,threshold,n_features,n_train,n_test,n_test_pMCI,...,auc,auc_pr,bal_acc,mcc,sens_pMCI,spec_sMCI,prec_pMCI,f1_pMCI,recall_pMCI,precision_pMCI
0,nested_cv_multimodel,1,RandomForestClassifier,0.754688,"{'model': 'RandomForestClassifier', 'max_depth...",0.519205,200,420,105,26,...,0.760954,0.509838,0.700341,0.362030,0.653846,0.746835,0.459459,0.539683,0.653846,0.459459
1,nested_cv_multimodel,2,RandomForestClassifier,0.773335,"{'model': 'RandomForestClassifier', 'max_depth...",0.403698,200,420,105,26,...,0.727361,0.436491,0.682327,0.315493,0.807692,0.556962,0.375000,0.512195,0.807692,0.375000
2,nested_cv_multimodel,3,RandomForestClassifier,0.758997,"{'model': 'RandomForestClassifier', 'max_depth...",0.492936,200,420,105,26,...,0.858325,0.642773,0.777264,0.490616,0.807692,0.746835,0.512195,0.626866,0.807692,0.512195
3,nested_cv_multimodel,4,RandomForestClassifier,0.724844,"{'model': 'RandomForestClassifier', 'max_depth...",0.444875,200,420,105,25,...,0.770000,0.506505,0.740000,0.409048,0.880000,0.600000,0.407407,0.556962,0.880000,0.407407
4,nested_cv_multimodel,5,RandomForestClassifier,0.771855,"{'model': 'RandomForestClassifier', 'max_depth...",0.507811,200,420,105,25,...,0.732000,0.582733,0.677500,0.307477,0.680000,0.675000,0.395349,0.500000,0.680000,0.395349



Resumo geral (média ± desvio entre folds):
  accuracy      : 0.690 ± 0.055
  auc           : 0.770 ± 0.053
  auc_pr        : 0.536 ± 0.079
  bal_acc       : 0.715 ± 0.042
  mcc           : 0.377 ± 0.075
  sens_pMCI     : 0.766 ± 0.095
  spec_sMCI     : 0.665 ± 0.086
  prec_pMCI     : 0.430 ± 0.056
  f1_pMCI       : 0.547 ± 0.050
  recall_pMCI   : 0.766 ± 0.095
  precision_pMCI: 0.430 ± 0.056

Modelo vencedor por fold:


,n_folds
best_model,
RandomForestClassifier,5



Média de AUC no teste por modelo vencedor:


,n_folds,auc_mean,auc_std
best_model,,,
RandomForestClassifier,5,0.769728,0.052785


# Rede neural


In [24]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

PATH = "csvs/abordagem_4_sMCI_pMCI_extremos/feat_merge_all_hipp_notext_filtered.csv"

df = pd.read_csv(PATH)

mask_smci = df[df["GROUP"] == 0] 
mask_pmci = df[df["GROUP"] == 1]

# mask_smci.count()
# mask_pmci.count()

mask_smci = mask_smci.iloc[0:128]#.count()

balanced_input = pd.concat([mask_smci, mask_pmci])

META_COLS = {"ID_PT", "GROUP", "SEX"}
feature_cols = [c for c in balanced_input.columns if c not in META_COLS]  # 90 colunas

all_float_values = balanced_input[feature_cols].astype(float).values
# --- dados ---
# X: (n_amostras, 90)
# y: rótulos categóricos, ex.: "sMCI" / "pMCI"
X = all_float_values  # 90 colunas
y_raw = balanced_input["GROUP"].values

# le = LabelEncoder()
y = y_raw  # sMCI=0, pMCI=1 (ou vice-versa)

# --- rede neural ---
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),   # 2 camadas ocultas
    activation="relu",
    solver="adam",
    alpha=1e-4,                     # regularização L2
    batch_size=32,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,            # para no conjunto de validação interno
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),     # essencial para MLP
    ("mlp", mlp),
])

# --- validação cruzada ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_proba = cross_val_predict(
    pipeline, X, y, cv=cv, method="predict_proba"
)[:, 1]  # probabilidade da classe positiva (pMCI)

y_pred = (y_proba >= 0.5).astype(int)

print("AUC:", roc_auc_score(y, y_proba))
print("Acurácia:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred, target_names=["sMCI", "pMCI"]))

# reduzir o número de features 

AUC: 0.66937255859375
Acurácia: 0.625
              precision    recall  f1-score   support

        sMCI       0.62      0.66      0.64       128
        pMCI       0.63      0.59      0.61       128

    accuracy                           0.62       256
   macro avg       0.63      0.62      0.62       256
weighted avg       0.63      0.62      0.62       256



In [26]:
import warnings
import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, accuracy_score, balanced_accuracy_score,
    matthews_corrcoef, classification_report, roc_curve, confusion_matrix,
)
from sklearn.model_selection import cross_val_predict

warnings.filterwarnings("ignore")

SEED = 42
PATH = "csvs/abordagem_4_sMCI_pMCI_extremos/feat_rad_all_hipp_notext_filtered.csv"

# --- carregar ---
df = pd.read_csv(PATH)
META_COLS = {"ID_PT", "GROUP", "SEX"}
feature_cols = [c for c in df.columns if c not in META_COLS]

X = df[feature_cols].astype(float).values
y = df["GROUP"].astype(int).values

print(f"Amostras: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Classes: 0={sum(y==0)}, 1={sum(y==1)}")

# --- filtro de correlação (do experimentos.ipynb) ---
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.to_keep_ = None

    def fit(self, X, y=None):
        xf = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        if xf.shape[0] < 2:
            self.to_keep_ = np.arange(X.shape[1], dtype=int)
            return self
        c = np.corrcoef(xf.T)
        keep = []
        for j in range(c.shape[0]):
            if all(not (np.isfinite(c[j, k]) and abs(c[j, k]) > self.threshold) for k in keep):
                keep.append(j)
        self.to_keep_ = np.asarray(keep, dtype=int)
        return self

    def transform(self, X):
        return X[:, self.to_keep_]

# --- pipeline ---
pipeline = ImbPipeline([
    ("corr_filter", CorrelationFilter(threshold=0.95)),
    ("var_filter", VarianceThreshold(threshold=0.01)),
    ("kbest", SelectKBest(score_func=mutual_info_classif, k=30)),
    ("scaler", StandardScaler()),
    ("sampler", RandomUnderSampler(random_state=SEED)),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        alpha=1e-3,
        batch_size=32,
        learning_rate="adaptive",
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=SEED,
    )),
])

# --- grid search (opcional, mas recomendado) ---
param_grid = {
    "kbest__k": [15, 20, 30, 50],
    "mlp__hidden_layer_sizes": [(16,), (32,), (32, 16)],
    "mlp__alpha": [1e-4, 1e-3],
}

def tune_youden(y_true, scores):
    fpr, tpr, thr = roc_curve(y_true, scores)
    return thr[np.argmax(tpr - fpr)]

K_OUT, K_IN = 5, 5
outer_cv = StratifiedKFold(K_OUT, shuffle=True, random_state=SEED)
aucs = []
oof_scores = np.zeros(len(y))
oof_preds = np.zeros(len(y), dtype=int)

for fold, (tr, te) in enumerate(outer_cv.split(X, y), 1):
    inner_cv = StratifiedKFold(K_IN, shuffle=True, random_state=SEED + fold)

    grid = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring="roc_auc", n_jobs=-1)
    grid.fit(X[tr], y[tr])
    model = grid.best_estimator_

    tr_scores = cross_val_predict(model, X[tr], y[tr], cv=inner_cv, method="predict_proba")[:, 1]
    thr = tune_youden(y[tr], tr_scores)

    te_scores = model.predict_proba(X[te])[:, 1]
    
    te_pred = (te_scores >= thr).astype(int)
    oof_scores[te] = te_scores
    oof_preds[te] = te_pred     
    
    auc = roc_auc_score(y[te], te_scores)
    aucs.append(auc)
    print(f"Fold {fold} | AUC teste: {auc:.3f} | {grid.best_params_}")

fpr, tpr, thr_all = roc_curve(y, oof_scores)
best_thr = thr_all[np.argmax(tpr - fpr)]

print(f"\nLimiar Youden: {best_thr:.4f}")
print(f"AUC:          {roc_auc_score(y, oof_scores):.4f}")
print(f"Acurácia:     {accuracy_score(y, oof_preds):.4f}")
print(f"Balanced Acc: {balanced_accuracy_score(y, oof_preds):.4f}")
print(f"MCC:          {matthews_corrcoef(y, oof_preds):.4f}")
print("\nMatriz de confusão:")
print(confusion_matrix(y, oof_preds))
print(classification_report(y, oof_preds, target_names=["sMCI", "pMCI"]))
print(f"\nAUC médio: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

Amostras: 525 | Features: 96
Classes: 0=397, 1=128


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=41. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=45. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=42. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=44. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate

Fold 1 | AUC teste: 0.645 | {'kbest__k': 50, 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (32,)}


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=45. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=44. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=45. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=48. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate

Fold 2 | AUC teste: 0.635 | {'kbest__k': 30, 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (16,)}


...

Fold 3 | AUC teste: 0.771 | {'kbest__k': 30, 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (16,)}


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=42. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=42. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=43. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=46. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate

Fold 4 | AUC teste: 0.623 | {'kbest__k': 20, 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (32, 16)}


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=46. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=45. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=45. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:776: UserWarning: k=50 is greater than n_features=45. All the features will be returned.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate

Fold 5 | AUC teste: 0.697 | {'kbest__k': 20, 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (32,)}

Limiar Youden: 0.5865
AUC:          0.6633
Acurácia:     0.5924
Balanced Acc: 0.6061
MCC:          0.1825

Matriz de confusão:
[[230 167]
 [ 47  81]]
              precision    recall  f1-score   support

        sMCI       0.83      0.58      0.68       397
        pMCI       0.33      0.63      0.43       128

    accuracy                           0.59       525
   macro avg       0.58      0.61      0.56       525
weighted avg       0.71      0.59      0.62       525


AUC médio: 0.674 ± 0.054


# Testes comparativos

In [27]:
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score, roc_curve,
)
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ConvergenceWarning.*")

SEED = 42
K_OUT, K_IN = 5, 5
PATH = "csvs/abordagem_4_sMCI_pMCI_extremos/feat_merge_all_hipp_notext_filtered.csv"
TEST_NAME = "nested_cv_4_models_advanced_selection"

METRIC_COLS = [
    "accuracy", "auc", "auc_pr", "bal_acc", "mcc",
    "sens_pMCI", "spec_sMCI", "f1_pMCI",
]

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90):
        self.threshold = threshold
        self.to_keep_ = None

    def fit(self, X, y=None):
        xf = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        if xf.shape[0] < 2:
            self.to_keep_ = np.arange(X.shape[1], dtype=int)
            return self
        c = np.corrcoef(xf.T)
        keep = []
        for j in range(c.shape[0]):
            if all(
                not (np.isfinite(c[j, k]) and abs(c[j, k]) > self.threshold)
                for k in keep
            ):
                keep.append(j)
        self.to_keep_ = np.asarray(keep, dtype=int)
        return self

    def transform(self, X):
        return X[:, self.to_keep_]

def fold_metrics(y_true, scores, pred) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "auc": float(roc_auc_score(y_true, scores)),
        "auc_pr": float(average_precision_score(y_true, scores)),
        "bal_acc": float(balanced_accuracy_score(y_true, pred)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "sens_pMCI": float(tp / (tp + fn)) if (tp + fn) else 0.0,
        "spec_sMCI": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "f1_pMCI": float(f1_score(y_true, pred, pos_label=1, zero_division=0)),
    }

def tune_youden_threshold(y_true, scores) -> float:
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(np.unique(y_true)) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y_true, scores)
    return float(thr[int(np.argmax(tpr - fpr))])

def get_scores(model, X):
    clf = model.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def count_features_after_preprocess(model, X):
    Xt = X
    for name, step in model.steps[:-1]:
        if hasattr(step, "transform") and name != "sampler":
            Xt = step.transform(Xt)
    return int(Xt.shape[1])

def simplify_best_params(best_params: dict) -> dict:
    out = {}
    for k, v in best_params.items():
        if k == "clf":
            out["model"] = v.__class__.__name__
        elif k == "feature_reducer":
            out["reducer"] = "off" if v == "passthrough" else v.__class__.__name__
        else:
            out[k.replace("clf__", "").replace("feature_reducer__", "reducer__")] = v
    return out

def make_l1_selector():
    return SelectFromModel(
        LinearSVC(
            penalty="l1", dual=False, class_weight="balanced",
            max_iter=200_000, random_state=SEED,
        ),
        max_features=40,
    )

PARAM_GRIDS = {
    "svm": [
        {
            "feature_reducer": [make_l1_selector()],
            "feature_reducer__estimator__C": [0.01, 0.1, 1.0],
            "clf": [SVC(probability=True, class_weight="balanced", random_state=SEED)],
            "clf__C": [0.1, 1.0, 10.0],
            "clf__gamma": ["scale", "auto"],
        },
        {
            "feature_reducer": [PCA(n_components=0.90, random_state=SEED)],
            "clf": [SVC(probability=True, class_weight="balanced", random_state=SEED)],
            "clf__C": [0.1, 1.0, 10.0],
            "clf__gamma": ["scale", "auto"],
        },
        {
            "feature_reducer": ["passthrough"],
            "clf": [SVC(probability=True, class_weight="balanced", random_state=SEED)],
            "clf__C": [0.1, 1.0, 10.0],
            "clf__gamma": ["scale", "auto"],
        },
    ],
    "rf": [
        {
            "feature_reducer": [make_l1_selector()],
            "feature_reducer__estimator__C": [0.01, 0.1, 1.0],
            "clf": [RandomForestClassifier(class_weight="balanced", random_state=SEED)],
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [5, 10],
            "clf__min_samples_split": [2, 5],
        },
        {
            "feature_reducer": ["passthrough"],
            "clf": [RandomForestClassifier(class_weight="balanced", random_state=SEED)],
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [5, 10],
            "clf__min_samples_split": [2, 5],
        },
    ],
    "xgb": [
        {
            "feature_reducer": [make_l1_selector()],
            "feature_reducer__estimator__C": [0.01, 0.1, 1.0],
            "clf": [XGBClassifier(eval_metric="logloss", random_state=SEED)],
            "clf__n_estimators": [100, 200],
            "clf__learning_rate": [0.01, 0.1],
            "clf__max_depth": [3, 5],
        },
        {
            "feature_reducer": ["passthrough"],
            "clf": [XGBClassifier(eval_metric="logloss", random_state=SEED)],
            "clf__n_estimators": [100, 200],
            "clf__learning_rate": [0.01, 0.1],
            "clf__max_depth": [3, 5],
        },
    ],
    "mlp": [
        {
            "feature_reducer": [make_l1_selector()],
            "feature_reducer__estimator__C": [0.01, 0.1, 1.0],
            "clf": [MLPClassifier(
                activation="relu", alpha=1e-3, batch_size=32,
                learning_rate="adaptive", max_iter=400, early_stopping=True,
                validation_fraction=0.15, random_state=SEED,
            )],
            "clf__hidden_layer_sizes": [(32, 16), (50,)],
        },
        {
            "feature_reducer": [PCA(n_components=0.90, random_state=SEED)],
            "clf": [MLPClassifier(
                activation="relu", alpha=1e-3, batch_size=32,
                learning_rate="adaptive", max_iter=400, early_stopping=True,
                validation_fraction=0.15, random_state=SEED,
            )],
            "clf__hidden_layer_sizes": [(32, 16), (50,)],
        },
        {
            "feature_reducer": ["passthrough"],
            "clf": [MLPClassifier(
                activation="relu", alpha=1e-3, batch_size=32,
                learning_rate="adaptive", max_iter=400, early_stopping=True,
                validation_fraction=0.15, random_state=SEED,
            )],
            "clf__hidden_layer_sizes": [(32, 16), (50,)],
        },
    ],
}

def nested_cv_multimodel(X, y, param_grid, test_name: str = TEST_NAME):
    results = []
    base_pipeline = ImbPipeline([
        # ("corr_filter", CorrelationFilter(threshold=0.90)),
        # ("var_filter", VarianceThreshold(threshold=0.01)),
        ("scaler", StandardScaler()),
        ("sampler", RandomUnderSampler(random_state=SEED)),
        ("feature_reducer", "passthrough"),
        ("clf", SVC(random_state=SEED)),
    ])
    outer_cv = StratifiedKFold(K_OUT, shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        inner_cv = StratifiedKFold(K_IN, shuffle=True, random_state=SEED + fold)

        grid_search = GridSearchCV(
            base_pipeline, param_grid, cv=inner_cv,
            scoring="roc_auc", n_jobs=-1, refit=True,
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        model_name = best_model.named_steps["clf"].__class__.__name__

        method = "predict_proba" if hasattr(best_model.named_steps["clf"], "predict_proba") else "decision_function"
        train_scores = cross_val_predict(best_model, X_train, y_train, cv=inner_cv, method=method)
        if method == "predict_proba":
            train_scores = train_scores[:, 1]

        threshold = tune_youden_threshold(y_train, train_scores)
        test_scores = get_scores(best_model, X_test)
        test_preds = (test_scores >= threshold).astype(int)

        row = {
            "test": test_name,
            "fold": fold,
            "best_model": model_name,
            "best_inner_auc": float(grid_search.best_score_),
            "best_params": simplify_best_params(grid_search.best_params_),
            "threshold": threshold,
            "n_features": count_features_after_preprocess(best_model, X_train),
            **fold_metrics(y_test, test_scores, test_preds),
        }
        results.append(row)
        print(
            f"Fold {fold} | {model_name:22s} | "
            f"features={row['n_features']:3d} | "
            f"AUC inner={row['best_inner_auc']:.3f} | AUC test={row['auc']:.3f}"
        )

    return pd.DataFrame(results)

def print_summary(df_results, test_name):
    print(f"\n=== RESUMO ({test_name}) ===")
    for col in METRIC_COLS:
        print(f"  {col:14s}: {df_results[col].mean():.3f} ± {df_results[col].std():.3f}")

In [28]:
# ==========================================
# 1. FILTRAGEM GLOBAL (Fora do Loop - Ganho de Velocidade)
# ==========================================
if __name__ == "__main__":
    df = pd.read_csv(PATH)
    meta = ["ID_PT", "GROUP", "SEX"]
    X_raw = df.drop(columns=meta, errors='ignore').to_numpy(float)
    y = df["GROUP"].to_numpy(int)

    print(f"Atributos originais: {X_raw.shape[1]}")
    
    # Executa os filtros não-supervisionados apenas uma vez aqui fora
    c_filter = CorrelationFilter(threshold=0.90)
    X_filtered = c_filter.fit_transform(X_raw)
    
    v_filter = VarianceThreshold(threshold=0.01)
    X_stage1 = v_filter.fit_transform(X_filtered)
    
    print(f"Atributos prontos para o Nested CV: {X_stage1.shape[1]}")

    # ==========================================
    # 2. AJUSTE NO PIPELINE INTERNO
    # ==========================================
    # O pipeline base agora não recalcula correlação e o sampler foi para o fim
    base_pipeline = ImbPipeline([
        ("scaler", StandardScaler()),
        ("feature_reducer", "passthrough"), # PCA ou L1 olham para o treino completo
        ("sampler", RandomUnderSampler(random_state=SEED)), # Entra logo antes do modelo
        ("clf", SVC(random_state=SEED)),
    ])

    all_results = []
    for name in ["svm", "rf", "xgb", "mlp"]:
        print(f"\n{'='*40}\n{name.upper()}\n{'='*40}")
        df_res = nested_cv_multimodel(
            X_stage1, y,
            PARAM_GRIDS[name],
            test_name=f"nested_cv_{name}",
        )
        all_results.append(df_res)

    df_all = pd.concat(all_results, ignore_index=True)
    display(df_all.groupby("test")[METRIC_COLS].agg(["mean", "std"]))

Atributos originais: 156
Atributos prontos para o Nested CV: 59

SVM
Fold 1 | SVC                    | features= 16 | AUC inner=0.717 | AUC test=0.693
Fold 2 | SVC                    | features= 40 | AUC inner=0.727 | AUC test=0.713


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection

Fold 3 | SVC                    | features= 40 | AUC inner=0.708 | AUC test=0.779


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection

Fold 4 | SVC                    | features= 40 | AUC inner=0.683 | AUC test=0.746
Fold 5 | SVC                    | features= 22 | AUC inner=0.747 | AUC test=0.655

RF


...

/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(


Fold 4 | MLPClassifier          | features= 59 | AUC inner=0.647 | AUC test=0.816


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_base.py:116: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(


Fold 5 | MLPClassifier          | features= 18 | AUC inner=0.711 | AUC test=0.582


accuracy                 auc              auc_pr            \
                   mean       std      mean       std      mean       std   
test                                                                        
nested_cv_mlp  0.664762  0.058087  0.654376  0.096511  0.436300  0.082822   
nested_cv_rf   0.651429  0.078996  0.732812  0.058346  0.439141  0.042112   
nested_cv_svm  0.649524  0.020647  0.717401  0.047844  0.439600  0.042535   
nested_cv_xgb  0.626667  0.085290  0.695359  0.071083  0.423118  0.074625   

                bal_acc                 mcc           sens_pMCI            \
                   mean       std      mean       std      mean       std   
test                                                                        
nested_cv_mlp  0.620745  0.096656  0.219172  0.163014  0.533231  0.219495   
nested_cv_rf   0.687101  0.062779  0.327405  0.112894  0.756923  0.112901   
nested_cv_svm  0.661945  0.052987  0.282028  0.091097  0.686769  0.144229   
nested_cv_xgb  0.656170  0.038309  0.276626  0.064603  0.712308  0.123629   

              spec_sMCI             f1_pMCI            
                   mean       std      mean       std  
test                                                   
nested_cv_mlp  0.708259  0.082480  0.428263  0.104700  
nested_cv_rf   0.617278  0.112273  0.516884  0.077260  
nested_cv_svm  0.637120  0.049822  0.484959  0.054782  
nested_cv_xgb  0.600032  0.143963  0.483637  0.038376